In [1]:
import mwclient
import time

site = mwclient.Site('en.wikipedia.org', clients_useragent='MyBTCAnalysisProject/1.0 (contact: twoj_mail@example.com)')
page = site.pages["Bitcoin"]


In [2]:
revs = list(page.revisions(limit=10))

c:\Users\me505\Desktop\devops\bitcoin_prices_prediction\venv_btc_predictions\Lib\site-packages\mwclient\util.py:38: DeprecationWarning: limit and api_chunk_size both specified, this is not supported! limit is deprecated, will use value of api_chunk_size
  warnings.warn(


In [3]:
revs[0]

OrderedDict([('revid', 1352953263),
             ('parentid', 1352929374),
             ('user', 'Vgbyp'),
             ('timestamp',
              time.struct_time(tm_year=2026, tm_mon=5, tm_mday=7, tm_hour=6, tm_min=30, tm_sec=4, tm_wday=3, tm_yday=127, tm_isdst=-1)),
             ('comment',
              'Reverted 1 edit by [[Special:Contributions/Pinkyquecha|Pinkyquecha]] ([[User talk:Pinkyquecha|talk]]): Per [[WP:CRYSTAL]]')])

In [4]:
revs = sorted(revs, key=lambda rev: rev["timestamp"])

In [5]:
revs[0]

OrderedDict([('revid', 275832581),
             ('parentid', 0),
             ('user', 'Pratyeka'),
             ('timestamp',
              time.struct_time(tm_year=2009, tm_mon=3, tm_mday=8, tm_hour=16, tm_min=41, tm_sec=7, tm_wday=6, tm_yday=67, tm_isdst=-1)),
             ('comment', 'creation (stub)')])

In [6]:
import torch
import builtins

# To "wstrzykuje" torch wszędzie, nawet do wnętrza innych bibliotek
builtins.torch = torch 

from transformers import pipeline

# Teraz spróbuj uruchomić pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=-1)

def find_sentiment(text):
    sent = sentiment_pipeline([text[:250]])[0]
    score = sent["score"]
    if sent["label"] =="NEGATIVE":
        score *= -1
    return score

c:\Users\me505\Desktop\devops\bitcoin_prices_prediction\venv_btc_predictions\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\me505\Desktop\devops\bitcoin_prices_prediction\venv_btc_predictions\Lib\site-packages\huggingface_hub\file_download.py:1842: DeprecationWarning: hf_xet.download_files() is deprecated. Use XetSession().new_file_download_group().start_download_file() instead.
  xet_get(
c:\Users\me505\Desktop\devops\bitcoin_prices_prediction\venv_btc_predictions\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\me505\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degrad

In [11]:
find_sentiment("Bye")

-0.968428373336792

In [13]:
edits = {}

for rev in revs:
    date = time.strftime("%Y-%m-%d", rev["timestamp"])

    if date not in edits:
        edits[date] = dict(sentiments=list(), edit_count=0)

    edits[date]["edit_count"] += 1

    comment = rev.get("comment", "")
    edits[date]["sentiments"].append(find_sentiment(comment))

In [19]:
from statistics import mean

for key in edits:
    if len(edits[key]["sentiments"]) > 0:
        edits[key]["sentiment"] = mean(edits[key]["sentiments"])
        edits[key]["neg_sentiment"] = len([s for s in edits[key]["sentiments"] if s < 0]) / len(edits[key]["sentiments"])
    else: 
        edits[key]["sentiment"] = 0
        edits[key]["meg_sentiment"] = 0

    del edits[key]["sentiments"]

KeyError: 'sentiments'

In [20]:
edits

{'2009-03-08': {'edit_count': 4,
  'sentiment': -0.5505250245332718,
  'neg_sentiment': 0.75},
 '2009-08-05': {'edit_count': 1,
  'sentiment': 0.748120903968811,
  'neg_sentiment': 0.0},
 '2009-08-06': {'edit_count': 2,
  'sentiment': 0.9957457184791565,
  'neg_sentiment': 0.0},
 '2009-08-14': {'edit_count': 1,
  'sentiment': 0.930020809173584,
  'neg_sentiment': 0.0},
 '2009-10-13': {'edit_count': 2,
  'sentiment': -0.22750118374824524,
  'neg_sentiment': 0.5},
 '2009-11-18': {'edit_count': 1,
  'sentiment': 0.883950412273407,
  'neg_sentiment': 0.0},
 '2009-12-08': {'edit_count': 1,
  'sentiment': -0.9869275689125061,
  'neg_sentiment': 1.0},
 '2009-12-17': {'edit_count': 1,
  'sentiment': -0.9975171089172363,
  'neg_sentiment': 1.0},
 '2010-02-23': {'edit_count': 1,
  'sentiment': -0.9994946718215942,
  'neg_sentiment': 1.0},
 '2010-03-18': {'edit_count': 1,
  'sentiment': 0.8758773803710938,
  'neg_sentiment': 0.0},
 '2010-04-13': {'edit_count': 4,
  'sentiment': 0.8443556129932404

In [21]:
import pandas as pd

edits_df = pd.DataFrame.from_dict(edits, orient="index")

In [22]:
edits_df

,edit_count,sentiment,neg_sentiment
2009-03-08,4,-0.550525,0.75
2009-08-05,1,0.748121,0.00
2009-08-06,2,0.995746,0.00
2009-08-14,1,0.930021,0.00
2009-10-13,2,-0.227501,0.50
...,...,...,...
2026-04-10,1,-0.984083,1.00
2026-04-14,1,-0.959644,1.00
2026-04-20,1,0.995622,0.00
2026-05-02,1,-0.999585,1.00


In [23]:
edits_df.index = pd.to_datetime(edits_df.index)

In [25]:
from datetime import datetime
dates = pd.date_range(start="2009-03-08", end=datetime.today())

In [26]:
dates

DatetimeIndex(['2009-03-08', '2009-03-09', '2009-03-10', '2009-03-11',
               '2009-03-12', '2009-03-13', '2009-03-14', '2009-03-15',
               '2009-03-16', '2009-03-17',
               ...
               '2026-05-03', '2026-05-04', '2026-05-05', '2026-05-06',
               '2026-05-07', '2026-05-08', '2026-05-09', '2026-05-10',
               '2026-05-11', '2026-05-12'],
              dtype='datetime64[us]', length=6275, freq='D')

In [27]:
edits_df = edits_df.reindex(dates, fill_value=0)

In [28]:
edits_df

,edit_count,sentiment,neg_sentiment
2009-03-08,4,-0.550525,0.75
2009-03-09,0,0.000000,0.00
2009-03-10,0,0.000000,0.00
2009-03-11,0,0.000000,0.00
2009-03-12,0,0.000000,0.00
...,...,...,...
2026-05-08,0,0.000000,0.00
2026-05-09,0,0.000000,0.00
2026-05-10,0,0.000000,0.00
2026-05-11,0,0.000000,0.00


In [29]:
rolling_edits = edits_df.rolling(30).mean()

In [30]:
rolling_edits

,edit_count,sentiment,neg_sentiment
2009-03-08,NaN,NaN,NaN
2009-03-09,NaN,NaN,NaN
2009-03-10,NaN,NaN,NaN
2009-03-11,NaN,NaN,NaN
2009-03-12,NaN,NaN,NaN
...,...,...,...
2026-05-08,0.200000,-0.064791,0.116667
2026-05-09,0.200000,-0.064791,0.116667
2026-05-10,0.166667,-0.031988,0.083333
2026-05-11,0.166667,-0.031988,0.083333


In [31]:
rolling_edits = rolling_edits.dropna()
rolling_edits

,edit_count,sentiment,neg_sentiment
2009-04-06,0.133333,-0.018351,0.025000
2009-04-07,0.000000,0.000000,0.000000
2009-04-08,0.000000,0.000000,0.000000
2009-04-09,0.000000,0.000000,0.000000
2009-04-10,0.000000,0.000000,0.000000
...,...,...,...
2026-05-08,0.200000,-0.064791,0.116667
2026-05-09,0.200000,-0.064791,0.116667
2026-05-10,0.166667,-0.031988,0.083333
2026-05-11,0.166667,-0.031988,0.083333


In [32]:
rolling_edits.to_csv("wikipedia_edits.csv")

In [ ]:
#downloading BTC data 
